# 3.0b — Tenseurs : strides, vues et mémoire

**Navigation** : [<< 3.0 — Théorie de l'information](3.0-Theorie-Information.ipynb) · [3.1 — Rétropropagation >>](3.1-Retropropagation.ipynb) · [Feuille de route de la série](README.md)

Toute opération « sans copie » que l'on nous vend — transposée, découpage, diffusion (broadcast) — n'est pas de la magie : c'est une **arithmétique d'adresses**. Un tableau ne stocke ni lignes ni colonnes : il stocke **un buffer plat** plus une recette pour s'y promener — la forme (*shape*) et les **strides** (pas, en éléments chez TorchSharp et en octets chez NumPy, le long de chaque dimension).

**Comment lire ce carnet.** Chaque section suit le même mouvement : une construction mesurée, puis la lecture de sa sortie, puis ce que la mesure interdit d'affirmer. Les quatre exercices s'intercalent aux endroits où la main doit prendre le relais de l'œil — ils sont faisables sans quitter le carnet, et leurs cellules s'exécutent sans erreur même non complétés. Les valeurs d'horloge et d'octets citées dans la prose sont celles imprimées par les cellules de ce run : elles varient d'une machine à l'autre, et c'est voulu — le carnet enseigne des *rapports* mesurés (un facteur, un ratio de stockage), pas des constantes.

Ce carnet ouvre cette mécanique, dans la discipline de la série : **TorchSharp natif** pour le stockage et les vues, **puis un témoin NumPy** — le même contrat d'adresse, plus deux témoins propres à torch (`view` qui exige des strides compatibles, le gradient qui traverse les vues). Chaque construction par `as_strided` est vérifiée contre un calcul d'indice fait à la main ; chaque affirmation de coût (octets, temps) est **mesurée dans ce carnet**, pas annoncée.

**Objectifs** : calculer offsets et pas, reconnaître alias et copies, contrôler fenêtres et gradients. **Prérequis** : tableaux, C# et notions de dérivation. **Durée** : 90–120 minutes.

**Version jumelle** : [Python](3.0b-Tenseurs-Strides-Vues-Python.ipynb). Installer TorchSharp-cpu 0.107.0 et un CPython avec NumPy ; définir `PYTHONNET_PYDLL` vers sa bibliothèque partagée, et si nécessaire `PYTHONHOME` et `PYTHONPATH`. Python.NET n’est utilisé que pour les témoins NumPy.


In [1]:
#r "nuget: TorchSharp-cpu, 0.107.0"
#r "nuget: PythonNet, 3.0.5"
using System; using System.Linq; using System.Diagnostics;
using TorchSharp; using static TorchSharp.torch; using Python.Runtime;
torch.set_num_threads(1);
if (!PythonEngine.IsInitialized) {
    var dll=Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if(string.IsNullOrWhiteSpace(dll)) throw new Exception("Définir PYTHONNET_PYDLL vers la bibliothèque partagée de CPython.");
    Runtime.PythonDLL=dll; PythonEngine.Initialize(); PythonEngine.BeginAllowThreads();
}
using(Py.GIL()) { dynamic np=Py.Import("numpy"),sys=Py.Import("sys"); Console.WriteLine($".NET {Environment.Version} | CPython {sys.version} | TorchSharp {typeof(Tensor).Assembly.GetName().Version} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version} | NumPy {np.__version__}"); }
string V(long[] x)=>"("+string.Join(", ",x)+")";
long Bytes(Tensor x)=>x.storage<long>().Count*x.element_size();
void Show(string label,Tensor x) => Console.WriteLine($"{label}: forme {V(x.shape)} | strides éléments {V(x.stride())} | offset {x.storage_offset()} | valeurs [{string.Join(", ", x.contiguous().data<long>().ToArray())}] | stockage {Bytes(x)} octets");

Installed Packages PythonNet, 3.0.5 TorchSharp-cpu, 0.107.0

.NET 10.0.14 | CPython 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] | TorchSharp 0.107.0.0 | Python.NET 3.0.5.0 | NumPy 2.4.4


## 1. L'identité d'un tableau : un buffer, une forme, des pas

Créons un tableau banal et demandons-lui ses papiers : où sont ses données (le buffer), à quel élément il commence (l'*offset*), et quelle recette il suit pour se déplacer (les *strides*). Le stockage TorchSharp expose la bande plate ; `storage_offset()` indique le départ.

Trois interrogations distinctes sont ici emmêlées, et le reste du carnet les démêle une à une. **Où sont les valeurs ?** Dans le buffer plat, et nulle part ailleurs — la sortie montre les douze entiers rangés à la queue. **Comment lire ce plat comme une grille ?** Par la recette (forme, strides) : c'est une *convention de lecture*, pas une structure. **Qu'est-ce que modifier `a` modifie-t-il ?** Le buffer — donc tout autre tableau qui lit le même buffer (la section 6 en fait un piège, l'autograd de torch en fait un atout).

Un réflexe à installer dès maintenant : devant un tableau inconnu, demander `shape`, `strides` et `storage_offset()` et le stockage avant toute hypothèse sur ce qu'il coûte. Les trois réponses disent si les opérations suivantes seront des réécritures de contrat (gratuites) ou des copies (payées en octets et en temps).

In [2]:
var a=torch.arange(12,dtype:ScalarType.Int64).reshape(3,4);
Show("a",a); Console.WriteLine("Buffer plat : "+string.Join(", ",a.storage<long>().ToArray()));

a: forme (3, 4) | strides éléments (4, 1) | offset 0 | valeurs [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] | stockage 96 octets


Buffer plat : 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11


Le tableau 3×4 n'occupe aucune structure 2D en mémoire : c'est la bande plate `0..11`, et les strides `(4, 1)` éléments, soit `(32, 8)` octets, disent : *pour avancer d'une ligne, saute 32 octets (4 éléments) ; d'une colonne, 8 octets (1 élément)*. L'adresse de l'élément $(i, j)$ est donc une formule :

$$\text{offset} + i \cdot 32 + j \cdot 8 \quad \text{octets}$$

La lecture à faire de cette sortie est double. D'abord **l'unité** : TorchSharp compte les strides en *éléments* : `(4,1)` correspond à `(32,8)` octets pour des entiers de huit octets. Ensuite **l'ordre** : le premier stride (4 éléments) est plus grand que le second (1 élément), signe d'un rangement *par lignes* (ordre C, pour *row-major*). Un tableau rangé par colonnes (ordre F, comme Fortran) échangerait les deux rôles. La formule ci-dessus n'est donc pas une propriété de NumPy mais un choix de convention — et tout ce qui suit (contiguïté, `view`, im2col) se joue sur ce choix.

TorchSharp appelle `as_strided` la construction inverse : donner un buffer, une forme et des pas, et obtenir un tableau qui **lit ce buffer** selon cette recette. Construisons une matrice 2×3 dans le même buffer, avec des pas de notre choix, et vérifions la formule à la main, élément par élément.

In [3]:
var v=a.as_strided(new long[]{2,3},new long[]{2,1},0); Show("vue 2×3",v);
var buf=a.storage<long>(); bool ok=true; for(int i=0;i<2;i++) for(int j=0;j<3;j++) ok &= v.contiguous().data<long>().ToArray()[i*3+j]==buf[i*2+j];
if(!ok)throw new Exception("Indice incohérent"); Console.WriteLine("Les six indices calculés manuellement concordent.");

vue 2×3: forme (2, 3) | strides éléments (2, 1) | offset 0 | valeurs [0, 1, 2, 2, 3, 4] | stockage 96 octets


Les six indices calculés manuellement concordent.


La vue 2×3 n'a **rien copié** : ses six valeurs sont lues dans le buffer d'origine aux adresses dictées par nos pas. La vérification porte sur les **six éléments** : pour chacun, l'adresse calculée par la formule (offset + pas) a été confrontée à l'élément que la vue affiche — la recette et l'affichage coïncident partout, pas seulement sur un coin.

Ce que `as_strided` apporte ici, et ce qu'il ne vérifie **pas** : NumPy ne vérifie pas les bornes ; TorchSharp contrôle le stockage. Nos pas `(2, 1)` en éléments restaient dans le buffer ; des pas plus larges que la forme, ou une forme plus grande que ce que le buffer contient, fabriqueraient un tableau qui **lit hors du buffer** — valeurs parasites ou crash, sans avertissement en NumPy. C'est l'outil chirurgical du carnet : on ne l'emploie jamais sans la vérification à la main qui l'accompagne (exercice 1 ci-dessous).

C'est toute la mécanique — le reste du carnet l'exploite.

### Exercice 1 — l'indice linéaire, à la main

**Contexte** : avant tout geste de `as_strided`, savoir prédire ce que la recette lit.
**Objectif** : écrire `IndiceLineaire(strides, i, j)` qui renvoie l'indice (en **éléments**, pas en octets) que l'adresse $(i, j)$ désigne dans le buffer.


Le stub C# ci-dessous conserve la même consigne ; les contrôles conditionnels vérifient plusieurs recettes.


In [4]:
long? IndiceLineaire(long[] strides,long i,long j,long itemsize=8) { /* TODO étudiant : strides en octets. */ return null; }
if(IndiceLineaire(new long[]{16,8},1,2)==null) Console.WriteLine("Exercice 1 à compléter");
else { bool valid=IndiceLineaire(new long[]{16,8},1,2)==4 && IndiceLineaire(new long[]{32,16},2,1)==10 && IndiceLineaire(new long[]{12,4},3,2,4)==11 && IndiceLineaire(new long[]{24,8},0,0)==0; Console.WriteLine(valid?"Fixtures indice validées":"Réponse incorrecte"); }

Exercice 1 à compléter


## 2. Les vues gratuites : transposée, sous-blocs, dilatation, diffusion

La transposée de `a` coûte-t-elle une copie ? Mesurons : si les deux tableaux pointent vers la même mémoire, l'opération n'a fait qu'**échanger les pas**.

« Gratuit » veut dire ici : *aucun octet de données n'a bougé*. Ce n'est pas « rapide » dans l'absolu — une vue reste assujettie à l'ordre de lecture de son buffer, et la section 3 mesurera précisément quand cet ordre se paie. Mais la construction elle-même ne touche qu'un petit objet d'en-tête (forme, pas, offset) : pour un tableau d'un million d'éléments comme pour douze, le coût de la transposée est le même — celui d'une fiche, pas celui d'une copie. Une conséquence pratique de cette économie, à garder en tête pour la suite : les bibliothèques que le carnet mesure (`sliding_window_view`, `broadcast_to`, l'`expand` de torch) sont toutes des **fabriques de contrats** au sens strict — elles rendent un objet dont les trois champs (offset, forme, pas) ont été réécrits et dont le buffer n'a pas bougé. C'est pourquoi leur coût de construction ne croît pas avec la taille du tableau, et pourquoi les mesurer « à vide » (construire puis jeter) ne renseigne sur rien : tout leur effet se joue dans les lectures et écritures qu'elles orchestrent ensuite.

La transposée et le sous-bloc sont des vues : une mutation temporaire suivie d’une restauration démontre leur alias.

In [5]:
var at=a.transpose(0,1); Show("transposée",at); var ast=a.storage<long>(); var ats=at.storage<long>(); long old=ast[0]; ats[0]=99; bool alias=ast[0]==99; ast[0]=old; if(!alias)throw new Exception("Alias transposée");Console.WriteLine("Alias démontré par mutation temporaire et restauration : "+alias);

transposée: forme (4, 3) | strides éléments (1, 4) | offset 0 | valeurs [0, 4, 8, 1, 5, 9, 2, 6, 10, 3, 7, 11] | stockage 96 octets


Alias démontré par mutation temporaire et restauration : True


Alias vérifié par mutation, pas échangés : la transposée est une **vue**. Le sous-bloc et la dilatation obéissent à la même économie — un slicing ne fait qu'écrire de nouveaux strides (et un nouvel offset) :

La sortie du bloc se lit comme une phrase, mot à mot. `strides (4, 2)` éléments, soit `(32, 16)` octets : le pas de **ligne** reste 32 (les lignes du bloc sont encore des lignes de `a`), le pas de **colonne** double à 16 (on ne garde qu'une colonne sur deux). `offset : 4` éléments, soit 32 octets : la vue commence un entier-cible plus loin — 32 octets, soit exactement la ligne 0 sautée par le `1:` du slicing. Chaque morceau de la syntaxe de slicing se retrouve, un à un, dans le triplet (offset, strides) : *traduction*, pas transformation. C'est la lecture à faire mentalement devant n'importe quel découpage : demander le nouveau contrat d'adresse avant de supposer quoi que ce soit sur le résultat — partagé, contigu, accessible en écriture.

In [6]:
var bloc=a.as_strided(new long[]{2,2},new long[]{4,2},4);Show("bloc [1:, ::2]",bloc);
var bs=bloc.storage<long>(); long b0=ast[4];bs[4]=77;bool blockAlias=ast[4]==77;ast[4]=b0;Console.WriteLine("Alias du stockage complet : "+blockAlias);

bloc [1:, ::2]: forme (2, 2) | strides éléments (4, 2) | offset 4 | valeurs [4, 6, 8, 10] | stockage 96 octets


Alias du stockage complet : True


La sortie du bloc précédent donne la recette complète : le slicing `[1:, ::2]` a doublé le pas de colonne (8 → 16 octets) en gardant le pas de ligne. La dilatation sur les *deux* axes est la généralisation immédiate — et le moment de vérifier que la syntaxe et le contrat d'adresse disent deux fois la même chose.

### Exercice 2 — la dilatation, sans slicing

**Contexte** : `a[::2, ::2]` dilate par sucre syntaxique ; la même vue s'écrit à la main.
**Objectif** : construire par `as_strided` la vue qui ne garde que les lignes et colonnes **paires** de `a` (3×4 → 2×2), et vérifier l'égalité avec `a[::2, ::2]`.


Le stub C# ci-dessous conserve la même consigne ; les contrôles conditionnels vérifient plusieurs recettes.


In [7]:
Tensor? DilaterParStrides(Tensor arr) { /* TODO : vue paires, sans copie. */ return null; }
if(DilaterParStrides(a) is null)Console.WriteLine("Exercice 2 à compléter"); else { bool valid=true; foreach(var shape in new[]{new long[]{3,4},new long[]{4,5},new long[]{1,1}}) { using var t=torch.arange(shape[0]*shape[1],dtype:ScalarType.Int64).reshape(shape); using var d=DilaterParStrides(t); if(d is null){valid=false;continue;} var expected=Enumerable.Range(0,(int)shape[0]).Where(i=>i%2==0).SelectMany(i=>Enumerable.Range(0,(int)shape[1]).Where(j=>j%2==0).Select(j=>(long)i*shape[1]+j)).ToArray(); valid &= d.shape.SequenceEqual(new long[]{(shape[0]+1)/2,(shape[1]+1)/2}) && d.contiguous().data<long>().ToArray().SequenceEqual(expected); var ts=t.storage<long>();var ds=d.storage<long>();ds[0]=97;valid &= ts[0]==97; } Console.WriteLine(valid?"Fixtures dilatation validées":"Réponse incorrecte"); }

Exercice 2 à compléter



(1,7): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



La **diffusion** (broadcast) est le sommet de cette économie : `expand` fabrique un tableau *logiquement* plus grand dont un stride vaut **zéro** — répéter sans stocker. À l'inverse, `repeat` matérialise chaque copie. La différence se lit dans les octets :

Le stride nul `(0, 1)` éléments, soit `(0, 8)` octets est la signature : avancer le long de l'axe diffusé ne coûte *aucun* octet de déplacement — toutes les « lignes » du tableau logique sont la même adresse, relue. Les octets réels mesurés rendent le contraste net : la vue n'occupe que la ligne stockée (32 octets), le `repeat` occupe le tableau entier (96). Le contenu est identique, la mémoire ne l'est pas — et cette asymétrie est un choix d'ingénierie quotidien : diffuser pour calculer (gratuit), matérialiser pour stocker (payé). Dans un entraînement, la même alternative se présente à chaque biais ajouté à un batch : le laisser en stride nul jusqu'à la dernière addition, ou le fondre dans les poids.

In [8]:
var ligne=torch.arange(4,dtype:ScalarType.Int64);var diffuse=ligne.expand(3,4);var repete=ligne.repeat(3,1);Show("expand",diffuse);Show("repeat",repete);Console.WriteLine("Contenu égal : "+diffuse.eq(repete).all().item<bool>());
using(Py.GIL()){dynamic np=Py.Import("numpy");using PyObject n=np.arange(4,dtype:np.int64);using var shape=new PyTuple(new PyObject[]{new PyInt(3),new PyInt(4)});using PyObject broadcast=np.broadcast_to(n,shape);bool shared=(bool)np.shares_memory(n,broadcast);bool writable=(bool)((dynamic)broadcast).flags.writeable;Console.WriteLine($"Témoin NumPy broadcast_to : mémoire partagée={shared}, écriture permise={writable}");if(!shared||writable)throw new Exception("Contrat NumPy inattendu");}


expand: forme (3, 4) | strides éléments (0, 1) | offset 0 | valeurs [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3] | stockage 32 octets


repeat: forme (3, 4) | strides éléments (4, 1) | offset 0 | valeurs [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3] | stockage 96 octets


Contenu égal : True


Témoin NumPy broadcast_to : mémoire partagée=True, écriture permise=False


Même contenu, **trois fois moins d'octets** pour la version diffusée : la ligne 0 du tableau diffusé et sa ligne 2 sont *la même mémoire*, relue. La vue `np.broadcast_to` est en lecture seule par contrat : plusieurs indices désignent la même adresse.

Une vue à stride nul n'est pas intrinsèquement en lecture seule : une écriture scalaire dans un `expand` peut modifier plusieurs positions logiques. Les écritures sur des vues qui se chevauchent exigent donc de la prudence. Utiliser `copy()` ou `clone()` pour couper explicitement l'alias ; `ascontiguousarray` ou `contiguous()` peuvent réutiliser un stockage déjà contigu.

## 3. Contiguïté : le prix de l'ordre séquentiel

Un tableau est **contigu** (ordre C) quand sa recette permet un parcours séquentiel en ordre C : les axes de taille 1 sont ignorés et les tableaux vides sont contigus : parcourir `flat[k]` et `a[i, j]` dans le même ordre. La transposée, le sous-bloc, le stride nul brisent ce contrat. TorchSharp le dit avec `is_contiguous()`, et le coût se mesure :

La sortie drapeau les quatre cas : seul `a`, jamais découpé, reste contigu. Trois des quatre constructions « gratuites » de la section 2 le perdent — le prix caché de la gratuité en octets se paie ici, en ordre d'accès, et la mesure qui suit le quantifie.

In [9]:
foreach(var (nom,t) in new[]{("a",a),("transpose",at),("bloc",bloc),("expand",diffuse)})Console.WriteLine($"{nom} : contigu={t.is_contiguous()}, pas éléments={V(t.stride())}");
var single=a.as_strided(new long[]{1,3},new long[]{999,1},0);var empty=a.as_strided(new long[]{0,3},new long[]{999,0},0);Console.WriteLine($"Singleton et vide contigus : {single.is_contiguous()}, {empty.is_contiguous()}");if(!single.is_contiguous()||!empty.is_contiguous())throw new Exception("Contrat singleton/vide");


a : contigu=True, pas éléments=(4, 1)


transpose : contigu=False, pas éléments=(1, 4)


bloc : contigu=False, pas éléments=(4, 2)


expand : contigu=False, pas éléments=(0, 1)


Singleton et vide contigus : True, True


Les quatre drapeaux imprimés ci-dessus forment le corpus de test idéal : un contigu (`a`) et trois non-contigus, chacun pour une raison différente (pas échangés, pas dilaté, pas nul). La fonction demandée ci-dessous doit rendre ces quatre verdicts *sans jamais construire les tableaux* — à partir de la seule recette (forme, pas), comme le ferait un diagnostic en amont d'un appel de bibliothèque.

### Exercice 3 — prédire la contiguïté depuis la recette

**Contexte** : les kernels compilés (BLAS, CUDA) exigent souvent des blocs contigus ; savoir le lire AVANT l'erreur.
**Objectif** : écrire `EstContigue(shape, strides, itemsize)` qui rend `True` ssi la recette décrit un parcours contigu C, en ignorant les strides des axes de taille 1 et en acceptant tout tableau vide de rang cohérent — sans tableau, juste la recette.


Le stub C# ci-dessous conserve la même consigne ; les contrôles conditionnels vérifient plusieurs recettes.

Une recette dont les rangs diffèrent ou dont une dimension est négative est invalide : retourner faux.


In [10]:
bool? EstContigue(long[] shape,long[] strides,long itemsize=8) { /* TODO : strides en octets. */ return null; }
if(EstContigue(new long[]{3,4},new long[]{32,8})==null)Console.WriteLine("Exercice 3 à compléter");else { bool valid=EstContigue(new long[]{3,4},new long[]{32,8})==true && EstContigue(new long[]{3,4},new long[]{8,32})==false && EstContigue(new long[]{3,4},new long[]{32,0})==false && EstContigue(new long[]{1,3},new long[]{999,8})==true && EstContigue(new long[]{0,3},new long[]{999,0})==true && EstContigue(new long[]{3,1},new long[]{8,999})==true && EstContigue(new long[]{2,3},new long[]{32,8})==false && EstContigue(new long[]{2,3},new long[]{8})==false && EstContigue(new long[]{-1,3},new long[]{24,8})==false;Console.WriteLine(valid?"Fixtures contiguïté validées":"Réponse incorrecte"); }

Exercice 3 à compléter


Pourquoi cet ordre importe : le matériel ne lit pas des éléments, il lit des **lignes de cache** de 64 octets — huit `float64` chargés d'un bloc, utiles ou non. Le contraste honnête n'est pas « la transposée est plus lente » (l'itérateur de NumPy réordonne si bien les boucles qu'une transposée peut paraître gratuite à l'horloge), c'est le **débit par élément utile** : sommons le buffer entier, puis un élément sur huit — un seul élément utile par ligne de 64 octets — sur un buffer de 128 Mo ; la taille du cache et son état dépendent de la machine :

Deux choix de protocole méritent d'être lus dans le code avant la sortie. Le buffer fait 128 Mo : cette taille ne garantit ni dépassement du cache L3 ni accès en mémoire vive à chaque lecture. Et la mesure retient le **meilleur** temps sur cinq essais : c’est le meilleur résultat observé, influencé par le cache, l’ordonnanceur et les autres activités. La vue `gros[:, ::8]` ne stocke rien de plus que `gros` — la somme parcourt 2 M d'éléments *utiles*, avec un pas de 64 octets ; le trafic exact dépend notamment du matériel et de son cache.
Le benchmark appelle NumPy depuis C# ; les durées incluent la traversée Python.NET et la consommation de la somme.


In [11]:
using(Py.GIL()) {
 dynamic np=Py.Import("numpy"); using PyObject big=np.arange(16000000,dtype:np.float64); using PyObject matrix=((dynamic)big).reshape(4000,4000);
 dynamic builtins=Py.Import("builtins"); using PyObject allSlice=builtins.slice(0,4000,1); using PyObject stepSlice=builtins.slice(0,4000,8); using var selection=new PyTuple(new PyObject[]{allSlice,stepSlice}); using PyObject sub=matrix.GetItem(selection);
 double Best(PyObject x,int count,long n,double expected) {double best=double.PositiveInfinity;for(int i=0;i<count;i++){var sw=Stopwatch.StartNew();using PyObject sum=((dynamic)x).sum();double value=(double)((dynamic)sum);sw.Stop();if(!double.IsFinite(value)||value!=expected)throw new Exception("Somme invalide");best=Math.Min(best,sw.Elapsed.TotalSeconds);}return n/best/1e6;}
 double full=Best(matrix,5,16000000,127999992000000), sampled=Best(sub,5,2000000,15999992000000);
 Console.WriteLine($"NumPy : 16 M float64 (128 Mo), cinq essais, meilleur débit : plein {full:F1} M éléments/s ; 1/8 {sampled:F1} M éléments/s ; rapport {full/sampled:F2}. Appels Python.NET inclus.");
}

NumPy : 16 M float64 (128 Mo), cinq essais, meilleur débit : plein 992.4 M éléments/s ; 1/8 245.2 M éléments/s ; rapport 4.05. Appels Python.NET inclus.


<null>

La cellule affiche le débit par élément utile pour chaque parcours et leur rapport dans ce run. Le résultat dépend du cache, de l'itérateur et de la machine : aucune dominance n'est garantie. C'est la raison d'être de `.contiguous()` (fabriquer la copie bien rangée) et, dans le monde CUDA, l'obsession des kernels pour les tuiles contiguës.

Les préchargements et l'itération influencent le temps observé ; les cinq essais ne permettent pas d'isoler leur contribution. Entre les deux se loge toute l'ingénierie des accès mémoire — le facteur dépend de la taille du buffer, de la largeur du pas, de la machine ; c'est précisément pourquoi le carnet mesure au lieu d'affirmer. La leçon à emporter n'est pas un chiffre mais une posture : **devant une vue découpée qui traverse un grand tableau, demander le débit utile avant de blâmer l'algorithme**.

## 4. Parité PyTorch : le même contrat, deux témoins propres à torch

PyTorch applique le même contrat d'adresse — vérifions la parité des constructions, puis les deux comportements que NumPy n'a pas : `view` qui exige des strides compatibles, et le **gradient** qui traverse les vues.

La parité mesurée se lit sur une seule unité de comptage : torch compte ses strides **en éléments** (`(4, 1)`), NumPy en **octets** (`(32, 8)`) — même recette, échelles différentes, et l'erreur classique qui consiste à comparer les deux tuples bruts se voit ici désamorcée par la double impression. L’alias de la transposée est vérifié par mutation temporaire et restauration, sans attribuer une adresse de données à un handle.

In [12]:
using(Py.GIL()) {
 dynamic np=Py.Import("numpy"), tricks=Py.Import("numpy.lib.stride_tricks");using PyObject n=np.arange(12,dtype:np.int64);using var sh=new PyTuple(new PyObject[]{new PyInt(2),new PyInt(3)});using var st=new PyTuple(new PyObject[]{new PyInt(16),new PyInt(8)});using PyObject nv=tricks.as_strided(n,shape:sh,strides:st,writeable:false); using PyObject rows=((dynamic)nv).tolist();var vals=rows.As<long[][]>().SelectMany(x=>x).ToArray();bool same=vals.SequenceEqual(v.contiguous().data<long>().ToArray());bool shared=(bool)np.shares_memory(n,nv);if(!same||!shared)throw new Exception("NumPy/TorchSharp diffèrent");Console.WriteLine($"NumPy (16,8) octets = TorchSharp (2,1) éléments : {same} ; mémoire NumPy partagée : {shared}");
}

NumPy (16,8) octets = TorchSharp (2,1) éléments : True ; mémoire NumPy partagée : True


**Témoin n° 1 — `view` exige une forme compatible avec les strides.** `view` ne fait pas de copie : cette transposée précise échoue, mais certains tenseurs non contigus admettent une vue compatible. L'erreur est le comportement enseigné, pas un accident :

L'exception signale une incompatibilité des strides : la transposée `(4, 3)` aux pas `(1, 4)` ne peut pas être lue comme une réinterprétation plate du stockage, la forme demandée ne respecte pas ces pas. Et la mutation contrôlée complète la leçon : `reshape(2, 6)` **réussit** lui, en fabriquant la copie que `view` refuse — `copie sous-jacente : True`. Deux méthodes, deux contrats : `view` = réécriture du contrat d'adresse seulement ; `reshape` = vue si possible, copie sinon. Choisir l'une ou l'autre, c'est choisir si l'on veut être averti du coût ou l'absorber en silence.

In [13]:
bool threw=false;try{using var bad=at.view(2,6);}catch(Exception ex){threw=true;Console.WriteLine("view(2,6) sur la transposée : "+ex.GetType().Name);}if(!threw)throw new Exception("Échec attendu");
var resh=at.reshape(2,6);var rs=resh.storage<long>();long saved=rs[0];rs[0]=777;bool copied=ast[0]!=777;rs[0]=saved;Console.WriteLine("reshape de cette transposée copie : "+copied);
var sampled=a.as_strided(new long[]{3,2},new long[]{4,2},0);var flat=sampled.view(6);Console.WriteLine($"Sous-échantillonnage non contigu compatible view : {!sampled.is_contiguous()} ; valeurs [{string.Join(",",flat.contiguous().data<long>().ToArray())}]");

view(2,6) sur la transposée : ExternalException


reshape de cette transposée copie : True


Sous-échantillonnage non contigu compatible view : True ; valeurs [0,2,4,6,8,10]


**Témoin n° 2 — le coût mesuré de `.contiguous()`.** Dans cet exemple, la copie bien rangée a un prix en octets ; l'opération `expand` (le broadcast de torch, stride nul) montre l'économie symétrique :

Les trois lignes de la sortie racontent le même aller-retour que chez NumPy, en strides comptés en éléments : `expand` fabrique le stride nul `(0, 1)` et ne stocke que la ligne source (32 octets), `contiguous()` déplie ici tout (96 octets) ; un tenseur déjà contigu peut être réutilisé. La vue `expand` partage la ligne d’origine ; ici, `contiguous()` alloue un nouveau stockage de 96 octets. Un consommateur qui exige l’ordre séquentiel peut rendre cette copie utile.

In [14]:
var cont=diffuse.contiguous();Show("expand",diffuse);Show("contiguous",cont);Console.WriteLine("Valeurs égales : "+diffuse.eq(cont).all().item<bool>());var already=a.contiguous();var als=already.storage<long>();long was=ast[0];als[0]=101;Console.WriteLine("contiguous sur a déjà contigu peut partager : "+(ast[0]==101));ast[0]=was;

expand: forme (3, 4) | strides éléments (0, 1) | offset 0 | valeurs [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3] | stockage 32 octets


contiguous: forme (3, 4) | strides éléments (4, 1) | offset 0 | valeurs [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3] | stockage 96 octets


Valeurs égales : True


contiguous sur a déjà contigu peut partager : True


## 5. Fenêtres glissantes et im2col : la convolution cachée dans les strides

Extraire toutes les fenêtres de largeur `w` d'une séquence semble exiger une boucle et des copies. C'est un pur problème de strides : `w` adresses qui se chevauchent. TorchSharp fournit `unfold` — écrivons la version nue et vérifions l'égalité :

Les pas de la sortie `(1, 1)` éléments, soit `(8, 8)` octets sont la signature du chevauchement : avancer d'une **fenêtre** (8 octets) et avancer d'un **élément dans la fenêtre** (8 octets aussi) coûtent le même déplacement — les fenêtres glissent d'un élément à la fois, donc chaque valeur du buffer sert dans plusieurs fenêtres simultanément. L'égalité vérifiée contre `unfold` dit que le sucre officiel et la construction nue `as_strided` fabriquent le même contrat d'adresse — `unfold` exprime directement la fenêtre et vérifie sa taille et son pas ; `as_strided` vérifie également les bornes du stockage en TorchSharp.

In [15]:
var seq=torch.arange(8,dtype:ScalarType.Int64);var windows=seq.as_strided(new long[]{6,3},new long[]{1,1},0);var unfolded=seq.unfold(0,3,1);Show("fenêtres",windows);bool winOk=windows.eq(unfolded).all().item<bool>();for(int i=0;i<6;i++)for(int j=0;j<3;j++)winOk &= windows.contiguous().data<long>().ToArray()[i*3+j]==i+j;Console.WriteLine("Égalité unfold et oracle d'indices : "+winOk);if(!winOk)throw new Exception("Fenêtres");
using(Py.GIL()){dynamic np=Py.Import("numpy"),tricks=Py.Import("numpy.lib.stride_tricks");using PyObject n=np.arange(8,dtype:np.int64);using PyObject nw=tricks.sliding_window_view(n,3);using PyObject rows=((dynamic)nw).tolist();bool equal=rows.As<long[][]>().SelectMany(r=>r).SequenceEqual(windows.contiguous().data<long>().ToArray());if(!equal)throw new Exception("Fenêtres NumPy divergentes");Console.WriteLine("Témoin NumPy sliding_window_view réel : "+equal);}


fenêtres: forme (6, 3) | strides éléments (1, 1) | offset 0 | valeurs [0, 1, 2, 1, 2, 3, 2, 3, 4, 3, 4, 5, 4, 5, 6, 5, 6, 7] | stockage 64 octets


Égalité unfold et oracle d'indices : True


Témoin NumPy sliding_window_view réel : True


Six fenêtres de trois éléments : **18 valeurs logiques, 8 stockées**. Ce mécanisme est l'`im2col` de la convolution — déplier chaque patch 3×3 d'une image en colonnes pour n'avoir plus qu'un produit matriciel. Écrivons-le à la main et vérifions contre une double boucle :

La sortie 2D généralise la comptabilité : forme `(4, 4, 3, 3)` — seize positions (grille 4×4 des coins possibles sur une image 6×6 avec des patchs 3×3), chacune tenant un patch 3×3, soit 144 valeurs logiques lues dans 36 stockées. Le ratio mesuré (quatre lectures par octet stocké, ici) vient du recouvrement : les patchs voisins partagent leurs lignes entières. Et le premier patch imprimé — colonnes 0-2 des lignes 0-2, valeurs 0 à 14 espacées de 6 — montre la forme `(3, 3)` du patch telle que la vue la découpe. La vue décrit le dépliage ; un backend peut matérialiser une matrice pour GEMM ou choisir un autre algorithme.

In [16]:
var img=torch.arange(36,dtype:ScalarType.Int64).reshape(6,6);var patches=img.as_strided(new long[]{4,4,3,3},new long[]{6,1,6,1},0);Show("patchs 2D",patches);var pv=patches.contiguous().data<long>().ToArray();bool patchOk=true;for(int i=0;i<4;i++)for(int j=0;j<4;j++)for(int u=0;u<3;u++)for(int w=0;w<3;w++)patchOk &= pv[((i*4+j)*3+u)*3+w]==(i+u)*6+j+w;Console.WriteLine("16 patchs et double boucle : "+patchOk);if(!patchOk)throw new Exception("Patchs");

patchs 2D: forme (4, 4, 3, 3) | strides éléments (6, 1, 6, 1) | offset 0 | valeurs [0, 1, 2, 6, 7, 8, 12, 13, 14, 1, 2, 3, 7, 8, 9, 13, 14, 15, 2, 3, 4, 8, 9, 10, 14, 15, 16, 3, 4, 5, 9, 10, 11, 15, 16, 17, 6, 7, 8, 12, 13, 14, 18, 19, 20, 7, 8, 9, 13, 14, 15, 19, 20, 21, 8, 9, 10, 14, 15, 16, 20, 21, 22, 9, 10, 11, 15, 16, 17, 21, 22, 23, 12, 13, 14, 18, 19, 20, 24, 25, 26, 13, 14, 15, 19, 20, 21, 25, 26, 27, 14, 15, 16, 20, 21, 22, 26, 27, 28, 15, 16, 17, 21, 22, 23, 27, 28, 29, 18, 19, 20, 24, 25, 26, 30, 31, 32, 19, 20, 21, 25, 26, 27, 31, 32, 33, 20, 21, 22, 26, 27, 28, 32, 33, 34, 21, 22, 23, 27, 28, 29, 33, 34, 35] | stockage 288 octets


16 patchs et double boucle : True


Le dépliage 2D à pas 1 est mesuré ci-dessus : seize positions, 144 valeurs logiques, 36 stockées. Le pas `s` entre positions est la dernière variable de la recette — c'est lui qu'une coude convolutionnelle règle quand elle « saute » des positions, et la comptabilité du stockage change avec lui.

### Exercice 4 — des fenêtres 2D avec chevauchement choisi

**Contexte** : `im2col` à pas `s` (le *stride* de convolution) généralise la recette.
**Objectif** : écrire `Patchs2D(img, p, s)` qui extrait par `as_strided` tous les patchs `p×p` avec un décalage `s` entre patchs, vérifiable contre une double boucle.


Le stub C# ci-dessous conserve la même consigne ; les contrôles conditionnels vérifient plusieurs recettes.

Pour une image 2D, exiger `p > 0`, `s > 0` et `p` au plus égal à chaque dimension ; refuser les autres valeurs avant de construire une vue.


In [17]:
Tensor? Patchs2D(Tensor image,long p,long step) { /* TODO : vue sûre par as_strided. */ return null; }
if(Patchs2D(img,3,2) is null)Console.WriteLine("Exercice 4 à compléter"); else { bool valid=true;foreach(var (height,width,p,step) in new[]{(6L,6L,3L,2L),(5L,7L,2L,1L),(4L,5L,4L,3L)}) {using var source=torch.arange(height*width,dtype:ScalarType.Int64).reshape(height,width);using var got=Patchs2D(source,p,step);if(got is null){valid=false;continue;}long nr=(height-p)/step+1,nc=(width-p)/step+1;valid &= got.shape.SequenceEqual(new[]{nr,nc,p,p});var values=got.contiguous().data<long>().ToArray();for(long i=0;i<nr;i++)for(long j=0;j<nc;j++)for(long u=0;u<p;u++)for(long w=0;w<p;w++)valid &= values[((i*nc+j)*p+u)*p+w]==(i*step+u)*width+j*step+w;var ss=source.storage<long>();var gs=got.storage<long>();gs[0]=97;valid &= ss[0]==97;}foreach(var (badP,badStep) in new[]{(0L,1L),(3L,0L),(7L,1L)}) {try {using var invalid=Patchs2D(img,badP,badStep);valid=false;}catch(ArgumentException){}}Console.WriteLine(valid?"Fixtures patchs validées":"Réponse incorrecte");}

Exercice 4 à compléter



(1,7): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



Une vue de fenêtres décrit les indices employés dans certaines convolutions ou attentions locales. Un backend peut aussi matérialiser les données ou utiliser un autre algorithme.

L'exercice 4 ci-dessus ajoute le dernier degré de liberté — le pas `s` entre fenêtres, qui est le *stride* de convolution au sens des réseaux : la même recette, avec les pas de position multipliés par `s`. Complété, il donne la pièce maîtresse pour relire n'importe quelle couche convolutive comme un produit matriciel sur une vue.

## 6. Aliasing : le piège des mémoires partagées

Si deux tableaux partagent le buffer, **modifier l'un modifie l'autre**. Ce n'est pas un bug de NumPy — c'est la conséquence directe de la section 1, et un piège quotidien dès qu'on croit travailler sur une copie :

Le scénario est minimal à dessein : une « victime », une vue sur sa première ligne, une écriture `999` dans la vue. Le retour à la victime montre le `999` installé — la vue n'a jamais possédé de mémoire, elle a écrit *dans* le buffer de la victime, à l'offset que son contrat d'adresse désignait. Une mutation temporaire vérifie le partage de stockage. Le réflexe défensif tient en une question à se poser avant toute écriture : *ce tableau est-il une vue ?* — et, dans le doute, la copie explicite `clone()` coupe le lien ; `contiguous()` peut conserver un alias.

In [18]:
var victime=torch.arange(6,dtype:ScalarType.Int64).reshape(2,3);var vue=victime.as_strided(new long[]{3},new long[]{1},0);var vs=vue.storage<long>();vs[1]=999;Show("victime après écriture",victime);Console.WriteLine("La mutation 999 est visible dans le stockage partagé : "+(victime.storage<long>()[1]==999));

victime après écriture: forme (2, 3) | strides éléments (3, 1) | offset 0 | valeurs [0, 999, 2, 3, 4, 5] | stockage 48 octets


La mutation 999 est visible dans le stockage partagé : True


Et côté torch, la même mémoire traversée par l'**autograd** : le gradient suit les vues, parce qu'une vue est une opération (différentiable) sur le tenseur d'origine, pas un nouveau stockage :

Le gradient imprimé `[0., 2., 4., 6., 8., 10.]` se décode à la main depuis le code : la chaîne est `view(2,3)` puis `transpose(0,1)` puis `pow(2)` puis `sum()`, donc chaque élément de `x` n'a subi qu'une élévation au carré avant la somme — et la dérivée de $x^2$ est $2x$, qui pour $x = 0..5$ donne exactement les valeurs imprimées. La leçon tient au trajet, pas au chiffre : le gradient a été calculé **à travers** la vue et la transposée, tandis que le carré alloue son résultat — l'autograd traite les contrats d'adresse comme des opérations différentiables dont le gradient peut accumuler les contributions en cas de chevauchement. C'est ce qui rend économiquement possible le graphique de calcul des frameworks : des vues et des transposées traversées par la dérivation, sans copie de données pour ces vues.

In [19]:
var x=torch.arange(6,dtype:ScalarType.Float64).requires_grad_();var y=x.view(2,3).transpose(0,1).pow(2);y.sum().backward();var grad=x.grad!.data<double>().ToArray();Console.WriteLine("Gradient à travers vue et transposée : ["+string.Join(", ",grad)+"]");if(!grad.SequenceEqual(new double[]{0,2,4,6,8,10}))throw new Exception("Gradient inattendu");

Gradient à travers vue et transposée : [0, 2, 4, 6, 8, 10]


## 7. Synthèse — ce qu'on emporte

| Geste | Ce qu'il fait vraiment | Coût mesuré dans ce carnet |
|---|---|---|
| `T`, slicing, `[::2, ::2]` | réécrit (offset, strides), même buffer | 0 octet de données — alias vérifié par mutation |
| `broadcast_to` / `expand` | stride **nul** sur l'axe diffusé | une ligne stockée pour N logiques |
| `repeat` | matérialise chaque copie | N lignes stockées |
| `.contiguous()` | copie si nécessaire pour une disposition contiguë | ici 96 octets ; peut aussi réutiliser le stockage |
| `view` (torch) | exige des strides compatibles avec la forme | exception pour la transposée du §4 |
| fenêtres / `im2col` | strides qui se chevauchent | 18 logiques / 8 stockées (1D) · 144 / 36 (2D) |

**Trois idées à retenir.** Un tableau est un **contrat de lecture** (buffer, offset, forme, pas), pas une structure de données. Tout ce qui « ne copie pas » ne fait que réécrire ce contrat — gratuit en mémoire, parfois payé en **ordre d'accès** (le débit par élément mesuré en §3). Et le piège symétrique : deux tableaux, une mémoire — l'*aliasing* de la section 6, qui devient un atout quand l'autograd le traverse.

Le fil relierait presque chaque section à la suivante par une opposition : la gratuité (§2) a son prix caché (§3, l'ordre), le contrat unique a ses deux dialectes (§4, torch et NumPy), le chevauchement des fenêtres (§5) n'est que la mutualisation poussée de la même adresse, et l'aliasing (§6) est la face sombre de la même mutualisation. Une seule mécanique, quatre régimes d'emploi.

Le notebook suivant ouvre la mécanique qui exploite ces tenseurs pour apprendre : [3.1 — Rétropropagation](3.1-Retropropagation.ipynb).

**Exercices à retenir** : calculer plusieurs indices linéaires, construire une dilatation qui partage le stockage, prédire la contiguïté avec axes singleton et vides, et construire des patchs 2D à pas variable avec preuve d’alias.


### Vérifiez votre compréhension

1. Pourquoi une vue de144valeurs peut-elle partager un stockage de36entiers ?
2. Un tenseur non contigu interdit-il toujours `view` ?
3. Le benchmark de128Mo prouve-t-il un accès hors cache ?

<details>
<summary>Réponses</summary>

1. Plusieurs indices de patchs désignent le même entier ; le stockage compte les données physiques, la forme compte les positions logiques.
2. Non : les strides doivent être compatibles avec la nouvelle forme. Le sous-échantillon du §4 accepte une vue ; la transposée testée la refuse.
3. Non : la capacité et l’état des caches dépendent de la machine. Les débits utiles mesurés sur cinq essais n’isolent pas les causes matérielles.

</details>

### Pour aller plus loin

- [Contrat de `Tensor.view`](https://docs.pytorch.org/docs/2.10/generated/torch.Tensor.view.html) : condition de compatibilité des strides.
- [Risques de `numpy.as_strided`](https://numpy.org/doc/stable/reference/generated/numpy.lib.stride_tricks.as_strided.html) et [fenêtres sûres](https://numpy.org/doc/stable/reference/generated/numpy.lib.stride_tricks.sliding_window_view.html).
- [TorchSharp](https://github.com/dotnet/TorchSharp) pour les API .NET et le stockage natif ; [Python.NET](https://pythonnet.github.io/) pour le témoin NumPy.
